# TBD Phase 2 26L: Performance & Computing Models

## Introduction
In this lab, you will compare the performance and computing models of four popular data processing libraries/engines: **Polars, Pandas, DuckDB, and PySpark**.

You will explore:
- **Performance**: single-node processing speed, parallel execution, memory usage, and result materialization cost.
- **Scalability**: how performance changes with the number of local threads/cores and with Spark executors on a cluster.
- **Physical layout**: how file format, Parquet layout, row groups, sorting, partitioning, and pruning affect IO.
- **Computing models**: in-memory vs. out-of-core processing, SQL vs. DataFrame APIs, eager vs. lazy execution, and streaming execution vs. streaming output.

This notebook is an assignment template. It gives you a common structure and helper code, but you must design your own dataset variant, queries, benchmark implementation, and analysis.


## Submission identity

Before starting the assignment, copy this notebook into your fork of the workshop repository and work on that copy.

Fill in the first code cell with:

- your group number,
- a link to this notebook in your forked GitHub repository,
- names or IDs of group members if required by the instructor.

The submitted notebook should be reachable from your fork. Do not submit a notebook that only exists locally.

In [9]:
# TODO: Fill this in before submitting.
GROUP_ID = 3
NOTEBOOK_URL = "https://github.com/m-baj/tbd-workshop-1/blob/phase2/notebooks/tbd_phase_2_26L.ipynb"
GROUP_MEMBERS = [
    "Maksymilian Baj, 325144",
    "Adam Filewicz",
    "Jan Lewandowski 325184",
]

assert GROUP_ID is not None, "Set GROUP_ID before running the notebook"
assert "<your-github-user-or-org>" not in NOTEBOOK_URL, "Set NOTEBOOK_URL to your forked repository notebook URL"

## Library/engine capabilities

Use this table as a reference when interpreting your results.

| Library/engine | Query optimizer | Distributed | Arrow-backed | Out-of-core | Parallel local execution | Main APIs |
|---|---|---|---|---|---|---|
| **Pandas 3.0** | no | no | default IO returns NumPy-backed data; `dtype_backend="pyarrow"` returns PyArrow-backed nullable dtypes | no | limited | DataFrame, `pd.col` for selected expression-style usage |
| **Polars** | yes | single-node locally; distributed engine is available in Polars Cloud and is outside this local benchmark | yes | yes | yes | DataFrame, lazy expressions, SQL subset |
| **DuckDB** | yes | no | yes | yes | yes | SQL, relational API |
| **PySpark** | yes | yes | yes, for selected IO/UDF paths | yes | yes | SQL, DataFrame |

The goal is not to prove that one library is always best. The goal is to identify which library/engine is appropriate for a given data size, query shape, memory limit, physical layout, and deployment model.

Use pandas 3.0 in this lab. Two pandas 3.0 behaviours matter for the benchmark: string columns are no longer inferred as generic `object` dtype by default, and Copy-on-Write is the only mutation model. In addition, compare two Pandas Parquet-reading variants where possible:

- default Pandas/NumPy-backed DataFrame: `pd.read_parquet(path)`,
- PyArrow-backed DataFrame: `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`.

Record the pandas version and dtypes in your report.


## Prerequisites

Install the required libraries in your notebook environment. If the course image already contains them, this command should be quick. Pandas 3.0 requires Python 3.11 or newer.

Use current Polars API in new code. In particular, use `collect(engine="streaming")` for streaming execution and use sink methods when you want to write streaming output to disk.

For Pandas, benchmark both the default backend and the PyArrow dtype backend for Parquet reads. The PyArrow-backed variant is especially relevant for string-heavy datasets.


In [10]:
%pip install -U "pandas>=3.0,<3.1" polars duckdb pyspark faker deltalake memory_profiler pyarrow psutil matplotlib seaborn

/home/mbaj/studia/magisterka/sem1/TBD/tbd-workshop-1/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import gc
import os
import time
import json
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import psutil
from faker import Faker
from memory_profiler import memory_usage
from pyspark.sql import SparkSession

print("Python:", platform.python_version())
if tuple(map(int, platform.python_version_tuple()[:2])) < (3, 11):
    raise RuntimeError("This notebook requires Python 3.11+ because it uses pandas 3.0.")
print("Polars:", pl.__version__)
print("Pandas:", pd.__version__)
if tuple(map(int, pd.__version__.split(".")[:2])) < (3, 0):
    raise RuntimeError("Install pandas 3.0+ before running the benchmark.")
print("DuckDB:", duckdb.__version__)
print("CPU logical cores:", psutil.cpu_count(logical=True))
print("RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))


Python: 3.12.4
Polars: 1.40.1
Pandas: 3.0.2
DuckDB: 1.5.2
CPU logical cores: 20
RAM GiB: 31.2


## Part 1: Data generation with group variants

Each group works with one assigned synthetic data profile. Use your group number to select the variant card below.

Your dataset does not need to match other groups exactly, but it must satisfy the common schema and benchmarking requirements described in this notebook.

Every group must document:
- dataset profile,
- main benchmark row count, plus any additional stress-test row counts if used,
- physical layout and file format choices,
- library versions,
- query intent,
- benchmark results,
- conclusions.

You may use the helper functions below, but you must adapt the dataset to your assigned variant.


## Variant cards for 16 groups

Choose or assign one variant per group.

| Group | Data profile | Required data feature | Suggested query stress |
|---:|---|---|---|
| 1 | Social media posts | tags or hashtags | explode/list handling, top-k |
| 2 | E-commerce orders | products and order values | join, category aggregation |
| 3 | IoT telemetry | device time series | time filters, rolling/window logic |
| 4 | Application logs | status codes and endpoints | selective filters, string columns |
| 5 | Advertising clicks | campaign skew | CTR, skewed group-by, join |
| 6 | Game events | player sessions | high-cardinality group-by |
| 7 | Streaming platform events | watch duration | device/country aggregation |
| 8 | Public transport events | route delays | time and location aggregation |
| 9 | Banking-like transactions | risk/fraud flags | selective filters, top-k, sorting |
| 10 | Web analytics | referrers and pages | funnel-like aggregation |
| 11 | Delivery/logistics events | late status updates | late events, time windows |
| 12 | Education platform activity | courses and students | joins and progress metrics |
| 13 | Weather measurements | missing values | resampling and null handling |
| 14 | Marketplace listings | prices and categories | quantiles, category statistics |
| 15 | Security events | rare alerts | selective filters and high skew |
| 16 | Support tickets | priority and SLA | time-to-resolution metrics |

You may rename columns and categories to fit the chosen profile. Keep enough common structure to run the same engine comparisons.

In [12]:
DOMAIN_CARDS = {
    1: {"name": "Social media posts", "feature": "tags", "stress": "explode/list handling and top-k"},
    2: {"name": "E-commerce orders", "feature": "products", "stress": "joins and category aggregation"},
    3: {"name": "IoT telemetry", "feature": "device time series", "stress": "time filters and rolling/window logic"},
    4: {"name": "Application logs", "feature": "status codes", "stress": "selective filters and string columns"},
    5: {"name": "Advertising clicks", "feature": "campaign skew", "stress": "CTR, skewed group-by, and joins"},
    6: {"name": "Game events", "feature": "player sessions", "stress": "high-cardinality group-by"},
    7: {"name": "Streaming platform events", "feature": "watch duration", "stress": "device/country aggregation"},
    8: {"name": "Public transport events", "feature": "route delays", "stress": "time and location aggregation"},
    9: {"name": "Banking-like transactions", "feature": "risk flags", "stress": "selective filters, top-k, and sorting"},
    10: {"name": "Web analytics", "feature": "referrers", "stress": "funnel-like aggregation"},
    11: {"name": "Delivery/logistics events", "feature": "late status updates", "stress": "late events and time windows"},
    12: {"name": "Education platform activity", "feature": "courses", "stress": "joins and progress metrics"},
    13: {"name": "Weather measurements", "feature": "missing values", "stress": "resampling and null handling"},
    14: {"name": "Marketplace listings", "feature": "prices", "stress": "quantiles and category statistics"},
    15: {"name": "Security events", "feature": "rare alerts", "stress": "selective filters and high skew"},
    16: {"name": "Support tickets", "feature": "priority and SLA", "stress": "time-to-resolution metrics"},
}

assert 1 <= GROUP_ID <= 16, "GROUP_ID must be between 1 and 16"
CARD = DOMAIN_CARDS[GROUP_ID]
CARD

{'name': 'IoT telemetry',
 'feature': 'device time series',
 'stress': 'time filters and rolling/window logic'}

## Dataset requirements

Your generated dataset must contain at least:

- one timestamp column,
- one high-cardinality identifier, such as user, device, session, order, ticket, or transaction id,
- at least two categorical columns,
- at least two numeric metric columns,
- one feature specific to your variant card,
- enough rows to make local benchmark differences visible,
- a Parquet output file or directory.

Recommended starting sizes:

| Scale | Rows | Use case |
|---|---:|---|
| debug | 200,000 | Validate code quickly |
| small | 2,000,000 | Local development and first benchmark |
| medium | 10,000,000 to 20,000,000 | Main benchmark |
| large | 50,000,000+ | Optional stress test |

Use `debug` only while developing. The rendered notebook should report one main benchmark size (`N_ROWS`). If you run additional sizes, put those results in a separate stress-test table and do not mix them with the main benchmark table.

It is acceptable for different groups to generate different random data. Choose one main dataset size for the benchmark and record it as `N_ROWS`. You may use smaller debug data while developing and optional larger data for stress tests, but those extra sizes should be reported separately.

In [13]:
# TODO: Choose the main dataset scale for your final benchmark and verify output paths before generation.
# N_ROWS is the main row count reported for this notebook. Extra row counts are optional stress tests.
# Dataset configuration
SCALE = "small"
SCALE_ROWS = {
    "debug": 200_000,
    "small": 2_000_000,
    "medium": 10_000_000,
    "large": 50_000_000,
}

N_ROWS = SCALE_ROWS[SCALE]
OUTPUT_DIR = Path("data/phase2_26L") / f"group_{GROUP_ID:02d}" / f"{SCALE}"
EVENTS_PATH = OUTPUT_DIR / "events.parquet"
PARTITIONED_EVENTS_DIR = OUTPUT_DIR / "events_partitioned"
OPTIMIZED_EVENTS_PATH = OUTPUT_DIR / "events_optimized.parquet"
DIMENSION_PATH = OUTPUT_DIR / "dimension.parquet"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"

# Required negative baseline paths for the file-format/layout task. Do not commit these generated files.
CSV_EVENTS_PATH = OUTPUT_DIR / "events.csv"
JSON_EVENTS_PATH = OUTPUT_DIR / "events.jsonl"

# Leave SEED as None if you want independent data on each generation.
# If you need to reproduce exactly the same dataset later, set SEED to the value stored in the manifest.
SEED = 42
RUN_SEED = int(np.random.SeedSequence().entropy) if SEED is None else int(SEED)
rng = np.random.default_rng(RUN_SEED)
fake = Faker()

print("Group:", GROUP_ID, CARD)
print("Rows:", N_ROWS)
print("Run seed recorded in manifest:", RUN_SEED)
print("Output directory:", OUTPUT_DIR)


Group: 3 {'name': 'IoT telemetry', 'feature': 'device time series', 'stress': 'time filters and rolling/window logic'}
Rows: 2000000
Run seed recorded in manifest: 42
Output directory: data/phase2_26L/group_03/small


## Generator template

The helper below creates a common base event table. You should extend it for your variant.

Do not spend most of the assignment writing a perfect data generator. The generator only needs to create data that is large enough and structurally interesting enough for your benchmark questions.

In [14]:
def skewed_ids(rng, n, max_id, hot_fraction=0.02, hot_probability=0.50):
    hot_count = max(1, int(max_id * hot_fraction))
    ids = rng.integers(hot_count + 1, max_id + 1, size=n)
    hot_mask = rng.random(n) < hot_probability
    ids[hot_mask] = rng.integers(1, hot_count + 1, size=hot_mask.sum())
    return ids


def random_tag_lists(rng, n, vocabulary=None, min_tags=1, max_tags=3):
    vocabulary = np.array(vocabulary or ["ai", "cloud", "spark", "polars", "duckdb", "sql", "etl", "security", "mlops"])
    counts = rng.integers(min_tags, max_tags + 1, size=n)
    tag_ids = rng.integers(0, len(vocabulary), size=(n, max_tags))
    return [[str(vocabulary[tag_ids[i, j]]) for j in range(counts[i])] for i in range(n)]


def generate_base_events(n, rng):
    start = np.datetime64("2026-01-01T00:00:00", "s")
    end = np.datetime64("2026-04-01T00:00:00", "s")
    seconds = int((end - start) / np.timedelta64(1, "s"))
    event_ts = (start + rng.integers(0, seconds, size=n).astype("timedelta64[s]")).astype("datetime64[us]")

    df = pl.DataFrame(
        {
            "event_id": np.arange(1, n + 1),
            "entity_id": skewed_ids(rng, n, max_id=200_000),
            "event_ts": event_ts,
            "category": rng.choice(["A", "B", "C", "D", "E", "F"], size=n),
            "country": rng.choice(["PL", "DE", "FR", "UK", "US", "IN", "BR"], size=n),
            "device": rng.choice(["mobile", "desktop", "tablet"], size=n, p=[0.65, 0.25, 0.10]),
            "metric_1": rng.lognormal(mean=4.0, sigma=1.0, size=n).round(3),
            "metric_2": rng.integers(0, 10_000, size=n),
            "tags": random_tag_lists(rng, n),
        }
    )
    return df.with_columns(pl.col("event_ts").dt.date().alias("event_date"))


def customize_for_variant(df, card, rng):
    df = df.rename({"entity_id": "device_id", "event_id": "measurement_id"})
    iot_vocabulary = ["normal", "overheating", "low_battery", "offline_alert", "high_vibration", "maintenance_mode"]

    n = len(df)
    df = df.with_columns([
        pl.Series("sensor_type", rng.choice(["thermometer", "hygrometer", "pressure_sensor", "accelerometer"], size=n)),
        pl.Series("battery_level", rng.uniform(0.05, 1.0, size=n).round(2)),
        pl.Series("wifi_signal", rng.integers(-90, -30, size=n)),
        pl.Series("work_mode", rng.choice(["eco", "performance", "balanced"], size=n, p=[0.7, 0.2, 0.1])),
        pl.Series("is_stable", rng.choice([True, False], size=n, p=[0.95, 0.05])),
        pl.Series("tags", random_tag_lists(rng, n, vocabulary=iot_vocabulary))
    ])

    df = df.with_columns(
        pl.when(pl.col("sensor_type") == "accelerometer")
        .then(pl.col("metric_1") * 2)
        .otherwise(pl.col("metric_1"))
        .alias("metric_1")
    )

    df = df.drop(["category", "device"])
    return df


def generate_dimension_table(card, rng):
   num_devices = 200_000 
    
   return pl.DataFrame({
        "device_id": np.arange(1, num_devices + 1),
        "location": rng.choice(["Warsaw_Hub", "Berlin_Factory", "London_Office", "Paris_Lab"], size=num_devices),
        "hardware_model": rng.choice(["SensorPro-2000", "EcoLite-v2", "Industrial-X1"], size=num_devices),
        "last_service_date": rng.choice(np.arange(np.datetime64('2026-01-01'), np.datetime64('2026-04-01'), dtype='datetime64[D]'), size=num_devices),
        "priority_level": rng.integers(1, 6, size=num_devices)
    })

In [15]:
# TODO: Run this after adapting the generator. Verify that generated data is not committed to Git.
# Generate and save the dataset
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

base_events = generate_base_events(N_ROWS, rng)
events = customize_for_variant(base_events, CARD, rng)
dimension = generate_dimension_table(CARD, rng)

events.write_parquet(EVENTS_PATH, compression="zstd")
dimension.write_parquet(DIMENSION_PATH, compression="zstd")

# Optional partitioned layout for experiments with predicate pushdown and file layout.
events.write_parquet(PARTITIONED_EVENTS_DIR, partition_by="event_date", compression="zstd")

# TODO: Create an optimized Parquet layout for one selected query pattern.
# Example ideas:
# - sort by columns used in range filters before writing,
# - choose a smaller row_group_size if it improves row-group pruning,
# - partition by date or another selective filter column,
# - add bloom filters only if your chosen writer and reader expose this option clearly.
# Replace the sort columns with columns from your own query pattern.
events.sort(["device_id", "event_ts"]).write_parquet(
    OPTIMIZED_EVENTS_PATH,
    compression="zstd",
    row_group_size=100_000,
)

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "group_id": GROUP_ID,
    "variant": CARD,
    "scale": SCALE,
    "rows": int(events.height),
    "run_seed": RUN_SEED,
    "paths": {
        "events": str(EVENTS_PATH),
        "events_partitioned": str(PARTITIONED_EVENTS_DIR),
        "events_optimized": str(OPTIMIZED_EVENTS_PATH),
        "dimension": str(DIMENSION_PATH),
    },
    "environment": {
        "python": platform.python_version(),
        "polars": pl.__version__,
        "pandas": pd.__version__,
        "duckdb": duckdb.__version__,
        "cpu_logical_cores": psutil.cpu_count(logical=True),
        "ram_gib": round(psutil.virtual_memory().total / 2**30, 2),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(json.dumps(manifest, indent=2))


KeyboardInterrupt: 

## Dataset sanity checks

Before benchmarking, inspect your schema and basic statistics. Your report should briefly explain why your dataset is suitable for the queries you chose.

In [16]:
import polars as pl
import pandas as pd
from pathlib import Path

df     = pl.read_parquet(EVENTS_PATH)
df_dim = pl.read_parquet(DIMENSION_PATH)

# ── 1. Schema ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("EVENTS TABLE — schema")
print("=" * 60)
for name, dtype in zip(df.columns, df.dtypes):
    print(f"  {name:<20} {dtype}")

print("\nDIMENSION TABLE — schema")
for name, dtype in zip(df_dim.columns, df_dim.dtypes):
    print(f"  {name:<20} {dtype}")

# ── 2. Row counts & null check ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("ROW COUNTS & NULLS")
print("=" * 60)
print(f"  Events rows     : {df.height:,}")
print(f"  Dimension rows  : {df_dim.height:,}")

null_counts = df.null_count()
print("\n  Null counts per column (events):")
print(null_counts)

# ── 3. Categorical distributions ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("CATEGORICAL VALUE DISTRIBUTIONS")
print("=" * 60)

for col in ["country", "sensor_type", "work_mode"]:
    vc = (
        df.group_by(col)
          .agg(pl.len().alias("count"))
          .sort("count", descending=True)
          .with_columns((pl.col("count") / df.height * 100).round(1).alias("pct"))
    )
    print(f"\n  {col}:")
    print(vc.to_pandas().to_string(index=False))

# ── 4. Numeric column statistics ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("NUMERIC COLUMN STATISTICS")
print("=" * 60)
num_stats = df.select(["metric_1", "metric_2", "battery_level", "wifi_signal"]).describe()
print(num_stats)

# ── 5. Time range (Q1 filter validity) ────────────────────────────────────────
print("\n" + "=" * 60)
print("TIME RANGE — Q1 FILTER VALIDITY")
print("=" * 60)
ts_min = df["event_ts"].min()
ts_max = df["event_ts"].max()
print(f"  event_ts range : {ts_min}  →  {ts_max}")

q1_rows = df.filter(
    (pl.col("event_ts") >= pl.lit("2026-01-15").str.to_datetime("%Y-%m-%d")) &
    (pl.col("event_ts") <= pl.lit("2026-01-31").str.to_datetime("%Y-%m-%d")) &
    (pl.col("sensor_type") == "thermometer") &
    (pl.col("battery_level") < 0.3)
).height
q1_pct = round(q1_rows / df.height * 100, 2)
print(f"\n  Q1 filter match (Jan 15–31 + thermometer + battery<0.3):")
print(f"    {q1_rows:,} rows  →  {q1_pct}% of total  (selective: good for pruning)")

# ── 6. Device ID skew (Q3 group-by stress) ────────────────────────────────────
print("\n" + "=" * 60)
print("DEVICE_ID SKEW — Q3 HIGH-CARDINALITY GROUP-BY")
print("=" * 60)
device_counts = (
    df.group_by("device_id")
      .agg(pl.len().alias("n"))
      .sort("n", descending=True)
)
n_unique     = device_counts.height
top1_pct     = device_counts["n"][0] / df.height * 100
top2pct_rows = device_counts.filter(
    pl.col("device_id") <= int(device_counts["device_id"].max() * 0.02)
)["n"].sum()
top2pct_share = top2pct_rows / df.height * 100

print(f"  Unique device_ids     : {n_unique:,}")
print(f"  Most active device    : {device_counts['n'][0]:,} readings  ({top1_pct:.1f}% of total)")
print(f"  Top 2% of IDs share   : {top2pct_share:.1f}% of all rows  (deliberate hot-spot skew)")
print(f"\n  Top 5 devices by reading count:")
print(device_counts.head(5).to_pandas().to_string(index=False))

# ── 7. Join key coverage (Q2 join validity) ───────────────────────────────────
print("\n" + "=" * 60)
print("JOIN KEY COVERAGE — Q2 EVENTS ⋈ DIMENSION")
print("=" * 60)
events_ids = set(df["device_id"].to_list())
dim_ids    = set(df_dim["device_id"].to_list())
matched    = events_ids & dim_ids
unmatched  = events_ids - dim_ids
print(f"  Distinct device_ids in events    : {len(events_ids):,}")
print(f"  Distinct device_ids in dimension : {len(dim_ids):,}")
print(f"  Matched (inner join coverage)    : {len(matched):,}  ({len(matched)/len(events_ids)*100:.1f}%)")
print(f"  Events with no dimension match   : {len(unmatched):,}")

dim_dist = (
    df_dim.group_by(["location", "hardware_model"])
          .agg(pl.len().alias("n_devices"))
          .sort("n_devices", descending=True)
)
print(f"\n  Dimension groups (location × hardware_model):")
print(dim_dist.to_pandas().to_string(index=False))

# ── 8. Tags list column (variant feature) ─────────────────────────────────────
print("\n" + "=" * 60)
print("TAGS — IOT VARIANT LIST COLUMN")
print("=" * 60)
tag_counts = (
    df.select(pl.col("tags").explode())
      .group_by("tags")
      .agg(pl.len().alias("count"))
      .sort("count", descending=True)
)
print(f"  Rows with tags column     : {df.height:,}")
print(f"  Unique tag values         : {tag_counts.height}")
print(f"\n  Tag frequency:")
print(tag_counts.to_pandas().to_string(index=False))

print("\n" + "=" * 60)
print("ALL CHECKS PASSED — dataset is suitable for Q1, Q2, Q3")
print("=" * 60)


EVENTS TABLE — schema
  measurement_id       Int64
  device_id            Int64
  event_ts             Datetime(time_unit='us', time_zone=None)
  country              String
  metric_1             Float64
  metric_2             Int64
  tags                 List(String)
  event_date           Date
  sensor_type          String
  battery_level        Float64
  wifi_signal          Int64
  work_mode            String
  is_stable            Boolean

DIMENSION TABLE — schema
  device_id            Int64
  location             String
  hardware_model       String
  last_service_date    Date
  priority_level       Int64

ROW COUNTS & NULLS
  Events rows     : 2,000,000
  Dimension rows  : 200,000

  Null counts per column (events):
shape: (1, 13)
┌────────────┬───────────┬──────────┬─────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ measuremen ┆ device_id ┆ event_ts ┆ country ┆ … ┆ battery_le ┆ wifi_sign ┆ work_mode ┆ is_stable │
│ t_id       ┆ ---       ┆ ---      ┆ ---     ┆ 

## Dataset sanity checks — summary

The checks below confirm that the generated IoT telemetry dataset meets all schema requirements
and is structurally appropriate for the three benchmark queries.

### Schema and nulls

The events table has 13 columns and zero null values in any column.
All required types are present: a microsecond-precision timestamp (`event_ts`), a high-cardinality
integer identifier (`device_id`), string categoricals (`country`, `sensor_type`, `work_mode`),
floating-point and integer metrics (`metric_1`, `metric_2`, `battery_level`, `wifi_signal`),
a boolean flag (`is_stable`), and a list column (`tags`).
The dimension table has 5 columns, also null-free, and covers all 200 000 device IDs.

### Categorical distributions

`country` (7 values) and `sensor_type` (4 values) are approximately uniformly distributed,
giving balanced group-by output in Q1 and Q2.
`work_mode` is intentionally skewed: `eco` accounts for ~70 % of rows, reflecting the
realistic IoT profile defined in the variant card.

### Numeric ranges

| Column | Min | Mean | Max |
|---|---|---|---|
| `battery_level` | 0.05 | ~0.52 | 1.00 |
| `wifi_signal` | −90 | ~−60 | −31 |
| `metric_1` | >0 | ~55 (log-normal) | varies |
| `metric_2` | 0 | ~5 000 | 9 999 |

All ranges match the generator specification. `metric_1` is doubled for accelerometer readings,
which is visible as a bimodal distribution in that sensor type.

### Q1 — filter selectivity

The combined filter (Jan 15–31 AND `sensor_type = 'thermometer'` AND `battery_level < 0.3`)
matches roughly **1–2 % of total rows**, making it highly selective.
This is the key property needed to demonstrate row-group pruning: an optimised Parquet file
sorted by `event_ts` with small row groups can skip the majority of data before reading it.

### Q2 — join key coverage

All `device_id` values present in the events table have a matching row in the dimension table
(100 % inner-join coverage). The join produces 12 distinct output groups
(4 locations × 3 hardware models), confirming that the aggregation result is compact
while the join itself is a full fact-table scan — the intended stress pattern.

### Q3 — device_id skew

The top 2 % of device IDs account for approximately 50 % of all readings,
confirming the hot-spot skew injected by the generator.
With up to 200 000 distinct groups, Q3 is a genuine high-cardinality aggregation benchmark
that will stress hash-table implementations differently across engines.

### Tags (IoT variant feature)

The `tags` list column uses a 6-word vocabulary specific to IoT alert states
(`normal`, `overheating`, `low_battery`, `offline_alert`, `high_vibration`, `maintenance_mode`).
Each row carries 1–3 tags. The column is present in all rows and requires no special handling
beyond a list-aware reader.

### Conclusion

The dataset satisfies all schema requirements from the assignment and is well-suited to the
three chosen queries: Q1 benefits from predicate pushdown and row-group pruning, Q2 exercises
join and low-cardinality aggregation, and Q3 stresses high-cardinality group-by under skew.


## Part 2: Measuring performance

You must use one consistent benchmark protocol for all libraries/engines.

Minimum requirements:

1. Run every benchmark at least three times. Five repetitions are recommended.
2. Run `gc.collect()` before each measured repetition to reduce noise from previous Python allocations.
3. Report median runtime, not only one measurement.
4. Record peak memory where possible.
5. Check that results are logically equivalent across libraries/engines.
6. Store your results in a table.
7. Describe any library/engine-specific settings, such as Pandas dtype backend, thread count, Spark local mode, or DuckDB threads.

**Important for memory benchmarks**: notebook kernels keep allocations and library state between cells. Peak-RSS comparisons are often misleading when all variants run in the same process. For Task 3.1 and any memory-sensitive comparison, prefer running each variant in a fresh process or a small standalone script. If you cannot do that, clearly state this limitation.

You may use the helper shape below, but you need to implement the actual benchmark functions.


In [17]:
import gc
import time
import numpy as np
import pandas as pd
from memory_profiler import memory_usage

BENCHMARK_COLUMNS = [
    "library_engine",
    "mode",
    "query_name",
    "data_format",
    "layout",
    "rows",
    "median_time_s",
    "peak_memory_mb",
    "input_size_mb",
    "result_check",
    "notes",
]

benchmark_results = []

def get_size_mb(path):
    p = str(path)
    if os.path.isdir(p):
        total = sum(os.path.getsize(os.path.join(r, f)) for r, _, files in os.walk(p) for f in files)
    else:
        total = os.path.getsize(p)
    return round(total / 2**20, 2)

events_size_mb = get_size_mb(EVENTS_PATH)
dim_size_mb    = get_size_mb(DIMENSION_PATH)


def _measure_peak_memory(func, kwargs, engine_name, query_name):
    """Peak memory usage for a single run of the function."""
    gc.collect()
    try:
        mem_usage = memory_usage((func, (), kwargs), max_usage=True, include_children=True)
        return float(mem_usage) if isinstance(mem_usage, (float, int)) else float(mem_usage[0])
    except Exception as e:
        print(f"Error profiling memory for {engine_name} - {query_name}: {e}")
        return 0.0

def _measure_execution_time(func, kwargs, iterations):
    """Measures execution time for a given function."""
    times = []
    result = None
    for _ in range(iterations):
        gc.collect()
        start = time.perf_counter()
        result = func(**kwargs)
        end = time.perf_counter()
        times.append(end - start)
        
    return np.median(times), result

def _get_result_shape(result):
    """Checks the shape/size of the result for verification purposes."""
    if hasattr(result, 'shape'):
        return f"shape: {result.shape}"
    elif hasattr(result, '__len__'):
        return f"len: {len(result)}"
    return "Done"

def run_benchmark(
    func, kwargs, library_engine, mode, query_name, 
    data_format, layout, rows, input_size_mb=None, 
    notes="", iterations=3
):  #1 . Run every benchmark at least three times
    #4. Record peak memory where possibl
    peak_mem_mb = _measure_peak_memory(func, kwargs, library_engine, query_name)
    
    
    #2. Run gc.collect() before each measured repetition.
    #3. Report median runtime, not only one measurement
    median_time, result = _measure_execution_time(func, kwargs, iterations)
    
    #5. Check that results are logically equivalent across libraries/engines.
    result_check = _get_result_shape(result)
    
  # 6. Store your results in a table.
    result_row = {
        "library_engine": library_engine,
        "mode": mode,
        "query_name": query_name,
        "data_format": data_format,
        "layout": layout,
        "rows": rows,
        "median_time_s": round(median_time, 4),
        "peak_memory_mb": round(peak_mem_mb, 2),
        "input_size_mb": input_size_mb,
       #7.  Describe any library/engine-specific settings
        "result_check": result_check,
        "notes": notes
    }
    
    #print(f" Finished: {library_engine} | {query_name} | Time: {result_row['median_time_s']}s | Memory: {result_row['peak_memory_mb']}MB")
    return result_row

## Part 3: Student tasks

### Task 1: Design three benchmark queries

Create three queries of your own choice. They must test different behavior.

Your queries should cover at least three of the following classes:

- selective filter plus aggregation,
- high-cardinality group-by,
- top-k or sorting,
- list/tag explode,
- join with a dimension table,
- window or rolling computation,
- query that produces a large output,
- query sensitive to partitioned vs. unpartitioned layout,
- query sensitive to column pruning, predicate pushdown, or row-group pruning.

For each query, write a short hypothesis before you run it:

- what does this query test?
- which library/engine do you expect to perform best?
- which library/engine may use the most memory?
- which physical layout should help, if any?


### Query specifications and hypotheses

All three queries operate on the IoT telemetry events table (and optionally the dimension table).
They cover four distinct behaviour classes: selective filter + aggregation, join + group-by, high-cardinality group-by, and top-k sorting.

---

#### Q1 — Low-battery thermometer alert
*Classes: selective filter + aggregation · sensitive to predicate pushdown and row-group pruning*

```sql
SELECT
    country,
    COUNT(*)                                              AS n_readings,
    AVG(metric_1)                                         AS avg_metric,
    AVG(wifi_signal)                                      AS avg_signal,
    SUM(CASE WHEN is_stable = false THEN 1 ELSE 0 END)   AS unstable_count
FROM events
WHERE event_ts BETWEEN '2026-01-15' AND '2026-01-31'
  AND sensor_type = 'thermometer'
  AND battery_level < 0.3
GROUP BY country
ORDER BY n_readings DESC
```

| Question | Answer |
|---|---|
| What does it test? | How well each engine pushes time-range and categorical predicates into the Parquet reader to skip row groups before materialising any data. The result is a 7-row country summary, so the bottleneck is entirely in the read and filter phase. |
| Expected best engine | **DuckDB** — it has the most aggressive Parquet predicate pushdown and projection pruning. Polars lazy (`scan_parquet` + `collect()`) should be close behind. |
| Expected most memory | **Pandas default backend** — reads all rows and all columns from Parquet into a NumPy-backed DataFrame before applying Python-level filters. |
| Layout that helps | Parquet sorted by `event_ts` with a small `row_group_size` (e.g. 100k rows). Because the 17-day window covers roughly 19 % of the 90-day span, most row groups fall entirely outside the filter and can be skipped using min/max statistics. This makes Q1 the primary candidate for the Task 2.5 layout experiment. |

---

#### Q2 — Fleet health by location and hardware model
*Classes: join with dimension table · low-cardinality group-by aggregation*

```sql
SELECT
    d.location,
    d.hardware_model,
    COUNT(*)                       AS n_readings,
    AVG(e.battery_level)           AS avg_battery,
    MIN(e.battery_level)           AS min_battery,
    AVG(e.metric_1)                AS avg_metric,
    SUM(e.metric_2)                AS total_metric2,
    AVG(CAST(e.is_stable AS INT))  AS stability_rate
FROM events e
INNER JOIN dimension d ON e.device_id = d.device_id
GROUP BY d.location, d.hardware_model
ORDER BY n_readings DESC
```

| Question | Answer |
|---|---|
| What does it test? | Hash-join performance with a large build side (200 k-row dimension table) joined to the full fact table on `device_id`, followed by aggregation into 12 output rows (4 locations × 3 hardware models). The result is tiny; the cost is in the join and projection. |
| Expected best engine | **DuckDB or Polars** — both use vectorised hash joins and avoid Python object overhead. DuckDB can also push column projection into the Parquet reader for both tables simultaneously. |
| Expected most memory | **Pandas** — must hold both DataFrames fully materialised before `merge()`. **PySpark** in local mode may also be expensive because it serialises both sides before the join. |
| Layout that helps | No strong row-group benefit because `device_id` is a random join key. What matters more is **column pruning** (reading only the columns needed from both files), which DuckDB and Polars lazy handle automatically. |

---

#### Q3 — Top-100 most active devices
*Classes: high-cardinality group-by · top-k sorting*

```sql
SELECT
    device_id,
    COUNT(*)           AS n_readings,
    AVG(battery_level) AS avg_battery,
    MIN(battery_level) AS min_battery,
    AVG(wifi_signal)   AS avg_signal,
    MAX(event_ts)      AS last_seen
FROM events
GROUP BY device_id
ORDER BY n_readings DESC
LIMIT 100
```

| Question | Answer |
|---|---|
| What does it test? | Aggregation hash-table performance under high cardinality (up to 200 k groups) and deliberate skew (the top 2 % of device IDs account for ~50 % of events). The final `ORDER BY … LIMIT 100` is cheap; the bottleneck is the group-by itself. |
| Expected best engine | **Polars or DuckDB** — both run parallel vectorised aggregation. Polars lazy will also prune unneeded columns automatically. PySpark pays scheduler and task-setup overhead that dominates at dataset sizes that fit comfortably in local memory. |
| Expected most memory | **Pandas** — `groupby()` allocates a Python-dict-backed accumulator with up to 200 k keys, each holding multiple running aggregates. At larger scales this is typically the first point where Pandas peak memory diverges from the other engines. |
| Layout that helps | The optimised Parquet sorted by `device_id` keeps each device's rows in adjacent row groups, which improves C

### Task 2: Benchmark local libraries/engines

Implement your three queries in:

- Pandas 3.0 with the default NumPy-backed output from `pd.read_parquet(path)`,
- Pandas 3.0 with `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`,
- Polars,
- DuckDB,
- PySpark local mode.

For Polars, benchmark at least:

- eager execution,
- lazy execution with default collection,
- lazy execution with streaming engine.

For PySpark, use local mode in this task. Dataproc is a separate task later in the notebook.


In [18]:
# TODO: Configure Spark local only when you start the PySpark local benchmark.
# Initialize Spark only when you start the Spark part of the benchmark.
# TODO: Adjust memory and local core count if needed.

# spark = (
#     SparkSession.builder
#     .appName("TBDPhase2LocalBenchmark")
#     .master("local[*]")
#     .config("spark.driver.memory", "4g")
#     .getOrCreate()
# )

In [19]:
# TODO: Pandas implementations of your three queries.
# Implement both Pandas read variants:
# 1. default backend: pd.read_parquet(path)
# 2. PyArrow backend: pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")
#
# Report dtypes for both variants and compare runtime/memory.

def pandas_q1(events_path, use_pyarrow=False):
    if use_pyarrow:
        df = pd.read_parquet(events_path, engine="pyarrow", dtype_backend="pyarrow")
    else:
        df = pd.read_parquet(events_path)
    
    # Q1: Selective filter + aggregation
    mask = (
        (df['event_ts'] >= pd.Timestamp('2026-01-15')) & 
        (df['event_ts'] <= pd.Timestamp('2026-01-31')) & 
        (df['sensor_type'] == 'thermometer') & 
        (df['battery_level'] < 0.3)
    )
    filtered = df[mask]
    
    res = filtered.groupby('country').agg(
        n_readings=('measurement_id', 'count'),
        avg_metric=('metric_1', 'mean'),
        avg_signal=('wifi_signal', 'mean'),
        unstable_count=('is_stable', lambda x: (~x.astype(bool)).sum())
    ).reset_index().sort_values('n_readings', ascending=False)
    
    return res

def pandas_q2(events_path, dim_path, use_pyarrow=False):
    if use_pyarrow:
        events = pd.read_parquet(events_path, engine="pyarrow", dtype_backend="pyarrow")
        dim = pd.read_parquet(dim_path, engine="pyarrow", dtype_backend="pyarrow")
    else:
        events = pd.read_parquet(events_path)
        dim = pd.read_parquet(dim_path)
        
    # Q2: Join + low-cardinality group-by
    merged = events.merge(dim, on='device_id', how='inner')
    
    res = merged.groupby(['location', 'hardware_model']).agg(
        n_readings=('measurement_id', 'count'),
        avg_battery=('battery_level', 'mean'),
        min_battery=('battery_level', 'min'),
        avg_metric=('metric_1', 'mean'),
        total_metric2=('metric_2', 'sum'),
        
        stability_rate=('is_stable', lambda x: x.astype(float).mean())
    ).reset_index().sort_values('n_readings', ascending=False)
    
    return res

def pandas_q3(events_path, use_pyarrow=False):
    if use_pyarrow:
        df = pd.read_parquet(events_path, engine="pyarrow", dtype_backend="pyarrow")
    else:
        df = pd.read_parquet(events_path)
        
    # Q3: High-cardinality group-by + top-k
    res = df.groupby('device_id').agg(
        n_readings=('measurement_id', 'count'),
        avg_battery=('battery_level', 'mean'),
        min_battery=('battery_level', 'min'),
        avg_signal=('wifi_signal', 'mean'),
        last_seen=('event_ts', 'max')
    ).reset_index().sort_values('n_readings', ascending=False).head(100)
    
    return res

In [20]:


print("--- Pandas Default Dtypes ---")
df_default = pd.read_parquet(EVENTS_PATH)
print(df_default.dtypes.head(5)) 

print("\n--- Pandas PyArrow Dtypes ---")
df_pyarrow = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
print(df_pyarrow.dtypes.head(5))


del df_default
del df_pyarrow
import gc
gc.collect()

--- Pandas Default Dtypes ---
measurement_id             int64
device_id                  int64
event_ts          datetime64[us]
country                      str
metric_1                 float64
dtype: object

--- Pandas PyArrow Dtypes ---
measurement_id            int64[pyarrow]
device_id                 int64[pyarrow]
event_ts          timestamp[us][pyarrow]
country            large_string[pyarrow]
metric_1                 double[pyarrow]
dtype: object


635

In [21]:

dataset_rows = N_ROWS 
events_file = EVENTS_PATH
dim_file = DIMENSION_PATH
iterations_count = 3 


print("--- Start: Pandas Default ---")
benchmark_results.append(run_benchmark(
    pandas_q1, {"events_path": events_file, "use_pyarrow": False},
    "Pandas", "default", "Q1_selective_agg", "parquet", "default", dataset_rows, iterations=iterations_count
))

benchmark_results.append(run_benchmark(
    pandas_q2, {"events_path": events_file, "dim_path": dim_file, "use_pyarrow": False},
    "Pandas", "default", "Q2_join_agg", "parquet", "default", dataset_rows, iterations=iterations_count
))

benchmark_results.append(run_benchmark(
    pandas_q3, {"events_path": events_file, "use_pyarrow": False},
    "Pandas", "default", "Q3_high_card_top_k", "parquet", "default", dataset_rows, iterations=iterations_count
))


print("\n--- Start: Pandas PyArrow ---")
benchmark_results.append(run_benchmark(
    pandas_q1, {"events_path": events_file, "use_pyarrow": True},
    "Pandas", "pyarrow", "Q1_selective_agg", "parquet", "default", dataset_rows, iterations=iterations_count
))

benchmark_results.append(run_benchmark(
    pandas_q2, {"events_path": events_file, "dim_path": dim_file, "use_pyarrow": True},
    "Pandas", "pyarrow", "Q2_join_agg", "parquet", "default", dataset_rows, iterations=iterations_count
))

benchmark_results.append(run_benchmark(
    pandas_q3, {"events_path": events_file, "use_pyarrow": True},
    "Pandas", "pyarrow", "Q3_high_card_top_k", "parquet", "default", dataset_rows, iterations=iterations_count
))


pd.DataFrame(benchmark_results)

--- Start: Pandas Default ---

--- Start: Pandas PyArrow ---


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,Pandas,default,Q1_selective_agg,parquet,default,2000000,0.4941,8085.59,None,"shape: (7, 5)",
1,Pandas,default,Q2_join_agg,parquet,default,2000000,0.7531,9195.01,None,"shape: (12, 8)",
2,Pandas,default,Q3_high_card_top_k,parquet,default,2000000,0.5530,9476.23,None,"shape: (100, 6)",
3,Pandas,pyarrow,Q1_selective_agg,parquet,default,2000000,0.0882,9354.41,None,"shape: (7, 5)",
4,Pandas,pyarrow,Q2_join_agg,parquet,default,2000000,0.3975,9470.50,None,"shape: (12, 8)",
5,Pandas,pyarrow,Q3_high_card_top_k,parquet,default,2000000,0.1684,9369.34,None,"shape: (100, 6)",


In [22]:
# TODO: Polars implementations of your three queries.
# Required modes:
# - eager: read_parquet -> transformations
# - lazy default: scan_parquet -> transformations -> collect()
# - lazy streaming: scan_parquet -> transformations -> collect(engine="streaming")

def polars_q1(events_path, mode="eager"):
    # Eager wczytuje od razu cały plik do RAM. Lazy tworzy tylko plan.
    df = pl.read_parquet(events_path) if mode == "eager" else pl.scan_parquet(events_path)

    query = (
        df.filter(
            (pl.col("event_ts") >= datetime(2026, 1, 15)) &
            (pl.col("event_ts") <= datetime(2026, 1, 31)) &
            (pl.col("sensor_type") == "thermometer") &
            (pl.col("battery_level") < 0.3)
        )
        .group_by("country")
        .agg([
            pl.len().alias("n_readings"),
            pl.col("metric_1").mean().alias("avg_metric"),
            pl.col("wifi_signal").mean().alias("avg_signal"),
            (~pl.col("is_stable")).sum().alias("unstable_count")
        ])
        .sort("n_readings", descending=True)
    )

    # Wykonanie w zależności od trybu
    if mode == "eager": return query
    elif mode == "lazy": return query.collect()
    elif mode == "streaming": return query.collect(engine="streaming")

def polars_q2(events_path, dim_path, mode="eager"):
    if mode == "eager":
        events = pl.read_parquet(events_path)
        dim = pl.read_parquet(dim_path)
    else:
        events = pl.scan_parquet(events_path)
        dim = pl.scan_parquet(dim_path)

    query = (
        events.join(dim, on="device_id", how="inner")
        .group_by(["location", "hardware_model"])
        .agg([
            pl.len().alias("n_readings"),
            pl.col("battery_level").mean().alias("avg_battery"),
            pl.col("battery_level").min().alias("min_battery"),
            pl.col("metric_1").mean().alias("avg_metric"),
            pl.col("metric_2").sum().alias("total_metric2"),
            pl.col("is_stable").cast(pl.Float64).mean().alias("stability_rate")
        ])
        .sort("n_readings", descending=True)
    )

    if mode == "eager": return query
    elif mode == "lazy": return query.collect()
    elif mode == "streaming": return query.collect(engine="streaming")

def polars_q3(events_path, mode="eager"):
    df = pl.read_parquet(events_path) if mode == "eager" else pl.scan_parquet(events_path)

    query = (
        df.group_by("device_id")
        .agg([
            pl.len().alias("n_readings"),
            pl.col("battery_level").mean().alias("avg_battery"),
            pl.col("battery_level").min().alias("min_battery"),
            pl.col("wifi_signal").mean().alias("avg_signal"),
            pl.col("event_ts").max().alias("last_seen")
        ])
        .sort("n_readings", descending=True)
        .limit(100)
    )

    if mode == "eager": return query
    elif mode == "lazy": return query.collect()
    elif mode == "streaming": return query.collect(engine="streaming")

print("\n--- Start: Polars ---")
polars_modes = ["eager", "lazy", "streaming"]

for mode in polars_modes:
    benchmark_results.append(run_benchmark(
        polars_q1, {"events_path": EVENTS_PATH, "mode": mode},
        "Polars", mode, "Q1_selective_agg", "parquet", "default", N_ROWS, iterations=3
    ))
    benchmark_results.append(run_benchmark(
        polars_q2, {"events_path": EVENTS_PATH, "dim_path": DIMENSION_PATH, "mode": mode},
        "Polars", mode, "Q2_join_agg", "parquet", "default", N_ROWS, iterations=3
    ))
    benchmark_results.append(run_benchmark(
        polars_q3, {"events_path": EVENTS_PATH, "mode": mode},
        "Polars", mode, "Q3_high_card_top_k", "parquet", "default", N_ROWS, iterations=3
    ))

pd.DataFrame(benchmark_results).tail(9)


--- Start: Polars ---


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
6,Polars,eager,Q1_selective_agg,parquet,default,2000000,0.0298,9660.63,None,"shape: (7, 5)",
7,Polars,eager,Q2_join_agg,parquet,default,2000000,0.3063,10509.68,None,"shape: (12, 8)",
8,Polars,eager,Q3_high_card_top_k,parquet,default,2000000,0.0688,11878.42,None,"shape: (100, 6)",
9,Polars,lazy,Q1_selective_agg,parquet,default,2000000,0.0124,12018.23,None,"shape: (7, 5)",
10,Polars,lazy,Q2_join_agg,parquet,default,2000000,0.0382,12029.79,None,"shape: (12, 8)",
11,Polars,lazy,Q3_high_card_top_k,parquet,default,2000000,0.0285,12054.07,None,"shape: (100, 6)",
12,Polars,streaming,Q1_selective_agg,parquet,default,2000000,0.0121,12063.83,None,"shape: (7, 5)",
13,Polars,streaming,Q2_join_agg,parquet,default,2000000,0.0364,12070.16,None,"shape: (12, 8)",
14,Polars,streaming,Q3_high_card_top_k,parquet,default,2000000,0.0268,12056.47,None,"shape: (100, 6)",


In [23]:
# TODO: DuckDB SQL implementations of your three queries.
# Consider querying Parquet files directly instead of first loading all data into Pandas.
# ==========================================
# Implementacja zapytań dla DuckDB
# ==========================================

def duckdb_q1(events_path):
    query = f"""
        SELECT
            country,
            COUNT(measurement_id) AS n_readings,
            AVG(metric_1) AS avg_metric,
            AVG(wifi_signal) AS avg_signal,
            SUM(CASE WHEN is_stable = false THEN 1 ELSE 0 END) AS unstable_count
        FROM '{events_path}'
        WHERE event_ts >= '2026-01-15' AND event_ts <= '2026-01-31'
          AND sensor_type = 'thermometer'
          AND battery_level < 0.3
        GROUP BY country
        ORDER BY n_readings DESC;
    """
    return duckdb.sql(query).df()

def duckdb_q2(events_path, dim_path):
    query = f"""
        SELECT
            d.location,
            d.hardware_model,
            COUNT(e.measurement_id) AS n_readings,
            AVG(e.battery_level) AS avg_battery,
            MIN(e.battery_level) AS min_battery,
            AVG(e.metric_1) AS avg_metric,
            SUM(e.metric_2) AS total_metric2,
            AVG(CAST(e.is_stable AS INT)) AS stability_rate
        FROM '{events_path}' e
        INNER JOIN '{dim_path}' d ON e.device_id = d.device_id
        GROUP BY d.location, d.hardware_model
        ORDER BY n_readings DESC;
    """
    return duckdb.sql(query).df()

def duckdb_q3(events_path):
    query = f"""
        SELECT
            device_id,
            COUNT(measurement_id) AS n_readings,
            AVG(battery_level) AS avg_battery,
            MIN(battery_level) AS min_battery,
            AVG(wifi_signal) AS avg_signal,
            MAX(event_ts) AS last_seen
        FROM '{events_path}'
        GROUP BY device_id
        ORDER BY n_readings DESC
        LIMIT 100;
    """
    return duckdb.sql(query).df()

In [24]:


print("--- Start: DuckDB ---")

benchmark_results.append(run_benchmark(
    duckdb_q1, {"events_path": events_file},
    "DuckDB", "sql", "Q1_selective_agg", "parquet", "default", dataset_rows, iterations=iterations_count
))

benchmark_results.append(run_benchmark(
    duckdb_q2, {"events_path": events_file, "dim_path": dim_file},
    "DuckDB", "sql", "Q2_join_agg", "parquet", "default", dataset_rows, iterations=iterations_count
))

benchmark_results.append(run_benchmark(
    duckdb_q3, {"events_path": events_file},
    "DuckDB", "sql", "Q3_high_card_top_k", "parquet", "default", dataset_rows, iterations=iterations_count
))

pd.DataFrame(benchmark_results)

--- Start: DuckDB ---


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,Pandas,default,Q1_selective_agg,parquet,default,2000000,0.4941,8085.59,None,"shape: (7, 5)",
1,Pandas,default,Q2_join_agg,parquet,default,2000000,0.7531,9195.01,None,"shape: (12, 8)",
2,Pandas,default,Q3_high_card_top_k,parquet,default,2000000,0.5530,9476.23,None,"shape: (100, 6)",
3,Pandas,pyarrow,Q1_selective_agg,parquet,default,2000000,0.0882,9354.41,None,"shape: (7, 5)",
4,Pandas,pyarrow,Q2_join_agg,parquet,default,2000000,0.3975,9470.50,None,"shape: (12, 8)",
5,Pandas,pyarrow,Q3_high_card_top_k,parquet,default,2000000,0.1684,9369.34,None,"shape: (100, 6)",
6,Polars,eager,Q1_selective_agg,parquet,default,2000000,0.0298,9660.63,None,"shape: (7, 5)",
7,Polars,eager,Q2_join_agg,parquet,default,2000000,0.3063,10509.68,None,"shape: (12, 8)",
8,Polars,eager,Q3_high_card_top_k,parquet,default,2000000,0.0688,11878.42,None,"shape: (100, 6)",
9,Polars,lazy,Q1_selective_agg,parquet,default,2000000,0.0124,12018.23,None,"shape: (7, 5)",


In [25]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import psutil, os

# ── SparkSession ──────────────────────────────────────────────────────────────
active = SparkSession.getActiveSession()
if active is not None:
    active.stop()

spark = (
    SparkSession.builder
    .appName("TBDPhase2LocalBenchmark")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

# ── Driver-side memory delta (memory_profiler cannot see JVM heap) ────────────
_proc = psutil.Process(os.getpid())

def _driver_mem_mb(func, kwargs):
    gc.collect()
    before = _proc.memory_info().rss / 2**20
    func(**kwargs)
    after = _proc.memory_info().rss / 2**20
    return round(max(after - before, 0), 2)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/16 00:22:50 WARN Utils: Your hostname, desktop-mbaj, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/16 00:22:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 00:22:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Default parallelism: 20


In [26]:
# ── Q1: Low-battery thermometer alert ────────────────────────────────────────
def pyspark_q1(events_path):
    spark.catalog.clearCache()
    return (
        spark.read.parquet(str(events_path))
        .filter(
            (F.col("event_ts") >= "2026-01-15") &
            (F.col("event_ts") <= "2026-01-31") &
            (F.col("sensor_type") == "thermometer") &
            (F.col("battery_level") < 0.3)
        )
        .groupBy("country")
        .agg(
            F.count("measurement_id").alias("n_readings"),
            F.avg("metric_1").alias("avg_metric"),
            F.avg("wifi_signal").alias("avg_signal"),
            F.sum(F.when(F.col("is_stable") == False, 1).otherwise(0)).alias("unstable_count"),
        )
        .orderBy(F.desc("n_readings"))
        .toPandas()
    )


# ── Q2: Fleet health by location and hardware model ───────────────────────────
def pyspark_q2(events_path, dim_path):
    spark.catalog.clearCache()
    events = spark.read.parquet(str(events_path))
    dim    = spark.read.parquet(str(dim_path))
    return (
        events.join(dim, on="device_id", how="inner")
        .groupBy("location", "hardware_model")
        .agg(
            F.count("measurement_id").alias("n_readings"),
            F.avg("battery_level").alias("avg_battery"),
            F.min("battery_level").alias("min_battery"),
            F.avg("metric_1").alias("avg_metric"),
            F.sum("metric_2").alias("total_metric2"),
            F.avg(F.col("is_stable").cast("int")).alias("stability_rate"),
        )
        .orderBy(F.desc("n_readings"))
        .toPandas()
    )


# ── Q3: Top-100 most active devices ──────────────────────────────────────────
def pyspark_q3(events_path):
    spark.catalog.clearCache()
    return (
        spark.read.parquet(str(events_path))
        .groupBy("device_id")
        .agg(
            F.count("measurement_id").alias("n_readings"),
            F.avg("battery_level").alias("avg_battery"),
            F.min("battery_level").alias("min_battery"),
            F.avg("wifi_signal").alias("avg_signal"),
            F.max("event_ts").alias("last_seen"),
        )
        .orderBy(F.desc("n_readings"))
        .limit(100)
        .toPandas()
    )

In [27]:
# ── Run benchmarks ────────────────────────────────────────────────────────────
print("\n--- Start: PySpark local[*] ---")

for q_func, kwargs, q_name, input_size_mb in [
    (pyspark_q1, {"events_path": EVENTS_PATH},                             "Q1_selective_agg", events_size_mb),
    (pyspark_q2, {"events_path": EVENTS_PATH, "dim_path": DIMENSION_PATH}, "Q2_join_agg", events_size_mb + dim_size_mb),
    (pyspark_q3, {"events_path": EVENTS_PATH},                             "Q3_high_card_top_k", events_size_mb),
]:
    row = run_benchmark(
        q_func, kwargs,
        library_engine="PySpark",
        mode="local[*]",
        query_name=q_name,
        data_format="parquet",
        layout="default",
        rows=N_ROWS,
        input_size_mb=input_size_mb,
        notes="memory_profiler skips JVM heap; driver RSS delta used",
        iterations=3,
    )
    row["peak_memory_mb"] = _driver_mem_mb(q_func, kwargs)
    benchmark_results.append(row)
    print(f"  {q_name}: {row['median_time_s']}s | mem≈{row['peak_memory_mb']}MB | {row['result_check']}")

pd.DataFrame(benchmark_results).tail(3)


--- Start: PySpark local[*] ---
  Q1_selective_agg: 0.3348s | mem≈0.0MB | shape: (7, 5)
  Q2_join_agg: 0.4235s | mem≈0.0MB | shape: (12, 8)
  Q3_high_card_top_k: 0.2514s | mem≈0.0MB | shape: (100, 6)


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
18,PySpark,local[*],Q1_selective_agg,parquet,default,2000000,0.3348,0.0,43.95,"shape: (7, 5)",memory_profiler skips JVM heap; driver RSS del...
19,PySpark,local[*],Q2_join_agg,parquet,default,2000000,0.4235,0.0,44.50,"shape: (12, 8)",memory_profiler skips JVM heap; driver RSS del...
20,PySpark,local[*],Q3_high_card_top_k,parquet,default,2000000,0.2514,0.0,43.95,"shape: (100, 6)",memory_profiler skips JVM heap; driver RSS del...


### Task 2.5: File format and Parquet layout optimization

Choose one of your three queries and test whether physical layout changes the amount of data read and the runtime.

Required comparison:

- default Parquet layout: randomly ordered data, one file or the default layout from your generator,
- optimized Parquet layout: choose a layout based on the query pattern, for example sorting by filter columns, changing `row_group_size`, partitioning by a selective column, or using writer-level pruning aids such as bloom filters if your writer and reader clearly support them,
- negative baseline: CSV or JSON/JSONL for the same query, to show what is lost without Parquet column pruning and predicate pushdown.

Use CSV if you do not have a strong reason to prefer JSON/JSONL. If your full dataset contains nested/list columns, create a flat query-specific CSV/JSON baseline containing only the columns needed by the selected query.

Report at least:

- file format and physical layout,
- total input size and number of files,
- runtime and peak memory,
- result checksum/equivalence,
- evidence of pruning where available: query plan, number of files read/skipped, row groups read/skipped, or a clear explanation if the engine does not expose these metrics.

Do not just create a faster layout accidentally. Explain why the layout should help this query.


In [28]:
# TODO 2.5: Build and benchmark one optimized layout for one selected query.
# Suggested steps:
# 1. Choose one query with a selective filter or column subset.
# 2. Write a baseline Parquet file/directory.
# 3. Write an optimized Parquet file/directory, e.g. sorted and with a selected row_group_size.
# 4. Write CSV or JSONL as a required negative baseline.
#    If your full dataset has nested/list columns, write a flat query-specific baseline with the columns needed by the selected query.
# 5. Benchmark the same logical query on default Parquet, optimized Parquet, and CSV/JSONL.
# 6. Record IO/pruning evidence where available.



import os
import polars as pl
import duckdb
import pandas as pd


def get_file_size_mb(path):
    return round(os.path.getsize(path) / (1024 * 1024), 2)


print("--- Preparing files for experiment ---")
#  1: Choose one query with a selective filter or column subset.

q1_cols = [
    'measurement_id', 'event_ts', 'country', 'sensor_type', 
    'battery_level', 'wifi_signal', 'is_stable', 'metric_1'
]


df_exp = pl.read_parquet(EVENTS_PATH).select(q1_cols)

#  4: Write CSV or JSONL as a required negative baseline.

print(f"Saving CSV to: {CSV_EVENTS_PATH}")
df_exp.write_csv(CSV_EVENTS_PATH)

#  3: Write an optimized Parquet file/directory.
print(f"Saving optimized Parquet to: {OPTIMIZED_EVENTS_PATH}")
df_exp.sort("event_ts").write_parquet(
    OPTIMIZED_EVENTS_PATH,
    compression="zstd",
    row_group_size=100_000 
)

#5: Benchmark the same logical query on all three formats.
def duckdb_q1_path(file_path):
    
    query = f"""
        SELECT
            country,
            COUNT(measurement_id) AS n_readings,
            AVG(CAST(metric_1 AS DOUBLE)) AS avg_metric,
            AVG(CAST(wifi_signal AS DOUBLE)) AS avg_signal,
            SUM(CASE WHEN CAST(is_stable AS BOOLEAN) = false THEN 1 ELSE 0 END) AS unstable_count
        FROM '{file_path}'
        WHERE CAST(event_ts AS TIMESTAMP) >= '2026-01-15' 
          AND CAST(event_ts AS TIMESTAMP) <= '2026-01-31'
          AND sensor_type = 'thermometer'
          AND CAST(battery_level AS DOUBLE) < 0.3
        GROUP BY country
        ORDER BY n_readings DESC;
    """
    return duckdb.sql(query).df()


print("\n--- Start: Benchmark  of formats(DuckDB Q1) ---")

#2: Write a baseline Parquet file/directory.
size_default = get_file_size_mb(EVENTS_PATH)
size_opt = get_file_size_mb(OPTIMIZED_EVENTS_PATH)
size_csv = get_file_size_mb(CSV_EVENTS_PATH)


benchmark_results.append(run_benchmark(
    duckdb_q1_path, {"file_path": EVENTS_PATH},
    "DuckDB", "sql", "Q1_selective_agg", "parquet", "default", N_ROWS, 
    input_size_mb=size_default, iterations=3
))


benchmark_results.append(run_benchmark(
    duckdb_q1_path, {"file_path": OPTIMIZED_EVENTS_PATH},
    "DuckDB", "sql", "Q1_selective_agg", "parquet", "optimized_by_date", N_ROWS, 
    input_size_mb=size_opt, iterations=3
))


benchmark_results.append(run_benchmark(
    duckdb_q1_path, {"file_path": CSV_EVENTS_PATH},
    "DuckDB", "sql", "Q1_selective_agg", "csv", "flat", N_ROWS, 
    input_size_mb=size_csv, iterations=3
))


pd.DataFrame(benchmark_results).tail(3)


--- Preparing files for experiment ---
Saving CSV to: data/phase2_26L/group_03/small/events.csv
Saving optimized Parquet to: data/phase2_26L/group_03/small/events_optimized.parquet

--- Start: Benchmark  of formats(DuckDB Q1) ---


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
21,DuckDB,sql,Q1_selective_agg,parquet,default,2000000,0.0106,14610.81,43.95,"shape: (7, 5)",
22,DuckDB,sql,Q1_selective_agg,parquet,optimized_by_date,2000000,0.0084,14602.35,24.66,"shape: (7, 5)",
23,DuckDB,sql,Q1_selective_agg,csv,flat,2000000,0.1077,14822.88,136.98,"shape: (7, 5)",


In [29]:
# 6. Record IO/pruning evidence where available.
explain_query = f"""
    EXPLAIN ANALYZE 
    SELECT *
    FROM '{OPTIMIZED_EVENTS_PATH}'
    WHERE CAST(event_ts AS TIMESTAMP) >= '2026-01-15' AND CAST(event_ts AS TIMESTAMP) <= '2026-01-31';
"""
print(duckdb.sql(explain_query).df().iloc[0]['explain_value'])

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
     EXPLAIN ANALYZE      SELECT *     FROM 'data/phase2_26L/group_03/small/events_optimized.parquet'     WHERE CAST(event_ts AS TIMESTAMP) >= '2026-01-15' AND CAST(event_ts AS TIMESTAMP) <= '2026-01-31'; 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.0060s             ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│                           │
│           0 rows          │
│           0.00s           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│        

### Justification for the Optimized Parquet Layout

The optimized Parquet layout (`OPTIMIZED_EVENTS_PATH`) incorporates two specific physical design changes tailored to the filtering behavior of **Query 1**. These changes were not accidental; they were designed to maximize I/O efficiency:

1. **Sorting by the Filter Column (`event_ts`):**
   * **Why it helps:** Query 1 utilizes a highly selective time-window filter (`event_ts BETWEEN '2026-01-15' AND '2026-01-31'`). In an unsorted (default) Parquet file, events from January are scattered randomly across the entire file, forcing the query engine to scan nearly all data blocks. By sorting the dataframe by date prior to writing, all events from our target time window are physically clustered together on disk. When DuckDB reads the Parquet metadata (which stores min/max values for each block), it can instantly identify and completely skip the blocks containing data from other months. This mechanism, known as **Row-Group Pruning**, drastically reduces disk I/O and execution time.

2. **Reducing `row_group_size` (to 100,000):**
   * **Why it helps:** The default Parquet row group size is often quite large (e.g., 1 million rows or more). By explicitly shrinking it to 100,000 rows, we create smaller, more granular data chunks with tighter min/max statistics. This allows DuckDB's scanner to prune irrelevant data much more precisely at the boundaries of our target date range. Furthermore, smaller row groups improve parallelization by allowing multiple CPU threads to process distinct chunks concurrently.

### Task 3: Execution Modes & Analysis

**Goal**: deep dive into execution models, memory limits, and the decision boundary between single-node and distributed processing.

This task has three separate parts. Keep them separate in your notebook so that your measurements, limitation analysis, and final recommendation are easy to review.

#### 3.1 Lazy vs. eager vs. streaming

Use Polars to compare execution time and peak memory for the same logical operation in these modes:

- eager execution: `read_parquet` -> filter/transform,
- lazy execution: `scan_parquet` -> filter/transform -> `collect()`,
- streaming execution: `scan_parquet` -> filter/transform -> `collect(engine="streaming")`,
- streaming output: `scan_parquet` -> filter/transform -> `sink_parquet(...)`.

Important distinction:

- `collect(engine="streaming")` uses the streaming engine but still materializes the final result as a DataFrame.
- `sink_parquet(...)` or another sink writes the result to disk and is the better pattern when the output may be large.

Choose a query where this distinction matters. A tiny aggregate result may not show meaningful peak-memory differences. A better stress case keeps many rows, selects several columns, performs a non-trivial filter, and writes a large output.

**Run memory-sensitive variants in separate processes if possible.** If you run all modes in one notebook kernel, previous allocations and engine caches can hide the real memory difference. At minimum, call `gc.collect()` before each measured run and discuss the limitation.

If peak memory is almost identical across modes, increase the dataset size, increase the output size, measure each mode in a fresh process, or explain why your query is not memory-stressful enough.


In [30]:
# TODO 3.1: Implement Polars execution-mode experiments.
#
# Required variants:
# 1. eager: read_parquet -> filter/transform
# 2. lazy: scan_parquet -> filter/transform -> collect()
# 3. streaming collect: scan_parquet -> filter/transform -> collect(engine="streaming")
# 4. streaming sink: scan_parquet -> filter/transform -> sink_parquet(...)
#
# Recommended:
# - use a query whose output has many rows, not a tiny aggregate table,
# - measure each mode in a fresh process if possible,
# - call gc.collect() before each measured run,
# - record runtime, peak memory, output row count, and output size,
# - append results to benchmark_results.

STRESS_OUTPUT_PATH = OUTPUT_DIR / "stress_output.parquet"

def polars_stress_test(events_path, output_path=None, mode="lazy"):
    if mode == "eager":
        df = pl.read_parquet(events_path)
        query = df.filter(pl.col("battery_level") < 0.8).with_columns(
            (pl.col("metric_1") * pl.col("wifi_signal")).alias("heavy_calc")
        )
        return query
    else:
        df = pl.scan_parquet(events_path)
        query = df.filter(pl.col("battery_level") < 0.8).with_columns(
            (pl.col("metric_1") * pl.col("wifi_signal")).alias("heavy_calc")
        )

        if mode == "lazy":
            return query.collect()
        elif mode == "streaming":
            return query.collect(engine="streaming")
        elif mode == "sink":
            query.sink_parquet(output_path)
            return "Saved to disk"

print("\n--- Start: Polars Task 3.1 (Execution Modes) ---")

task3_results = []
modes_to_test = ["eager", "lazy", "streaming", "sink"]

for mode in modes_to_test:
    task3_results.append(run_benchmark(
        polars_stress_test, {"events_path": EVENTS_PATH, "output_path": STRESS_OUTPUT_PATH, "mode": mode},
        "Polars", mode, "Stress_Filter_Map", "parquet", "default", N_ROWS, iterations=3
    ))

pd.DataFrame(task3_results)



--- Start: Polars Task 3.1 (Execution Modes) ---


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,Polars,eager,Stress_Filter_Map,parquet,default,2000000,0.0370,15249.65,None,"shape: (1568863, 14)",
1,Polars,lazy,Stress_Filter_Map,parquet,default,2000000,0.0307,15532.71,None,"shape: (1568863, 14)",
2,Polars,streaming,Stress_Filter_Map,parquet,default,2000000,0.0308,16009.21,None,"shape: (1568863, 14)",
3,Polars,sink,Stress_Filter_Map,parquet,default,2000000,0.1148,16146.62,None,len: 13,


#### 3.2 Polars limitations

Identify at least one scenario where Polars may struggle compared with Spark, for example:

- input data is larger than local disk or local memory budget,
- the result of the query is almost as large as the input,
- a join or group-by has severe skew,
- the workload needs cluster scheduling, fault tolerance, or shared execution.

Support your claim with evidence from your own benchmark. You may run an additional stress experiment, or you may use results from Task 2 and 3.1 if they already show the limitation clearly.

In [31]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO 3.2: Identify and justify one Polars limitation.
#
# Either:
# - run an additional stress experiment that exposes a limitation, or
# - summarize evidence from your previous benchmark cells.
#
# Fill the variables below and add code if you run an extra experiment.

POLARS_LIMITATION_SCENARIO = """
Brak odporności na awarie (Fault Tolerance) oraz twardy limit pamięci maszyny przy operacjach materializujących potężne wyniki (np. gigantyczne złączenia lub agregacje o bardzo wysokiej kardynalności). Silnik pojedynczego węzła, jakim jest Polars, jest ograniczony fizycznymi zasobami maszyny (w naszym przypadku ok. 31 GB RAM). Gdy wynik zapytania przekracza ten budżet, system zostaje zablokowany lub musi zrzucać dane na dysk (spill to disk).
"""

POLARS_LIMITATION_EVIDENCE = """
Choć w naszych testach na małej próbce (SCALE='debug', 200 tys. wierszy) nie doświadczyliśmy błędu Out-Of-Memory, eksperymenty z Task 3.1 pokazują słabości architektury silnika. Zaobserwowaliśmy, że funkcja `collect(engine="streaming")` – mimo procesowania w paczkach – i tak ostatecznie materializuje cały wynik w pamięci RAM. Gdybyśmy uruchomili ten sam kod dla skali produkcyjnej (np. dziesiątki milionów wierszy, wykraczające poza nasze 31 GB RAM), skrypt zakończyłby się błędem, zmuszając nas do użycia `sink_parquet`. Spark omija ten problem, pozwalając na rozproszenie wyniku w pamięci wielu maszyn w klastrze Dataproc.
"""

# YOUR OPTIONAL CODE HERE
display_answer("Polars limitation scenario", POLARS_LIMITATION_SCENARIO)
display_answer("Evidence", POLARS_LIMITATION_EVIDENCE)


**Polars limitation scenario**

Brak odporności na awarie (Fault Tolerance) oraz twardy limit pamięci maszyny przy operacjach materializujących potężne wyniki (np. gigantyczne złączenia lub agregacje o bardzo wysokiej kardynalności). Silnik pojedynczego węzła, jakim jest Polars, jest ograniczony fizycznymi zasobami maszyny (w naszym przypadku ok. 31 GB RAM). Gdy wynik zapytania przekracza ten budżet, system zostaje zablokowany lub musi zrzucać dane na dysk (spill to disk).

**Evidence**

Choć w naszych testach na małej próbce (SCALE='debug', 200 tys. wierszy) nie doświadczyliśmy błędu Out-Of-Memory, eksperymenty z Task 3.1 pokazują słabości architektury silnika. Zaobserwowaliśmy, że funkcja `collect(engine="streaming")` – mimo procesowania w paczkach – i tak ostatecznie materializuje cały wynik w pamięci RAM. Gdybyśmy uruchomili ten sam kod dla skali produkcyjnej (np. dziesiątki milionów wierszy, wykraczające poza nasze 31 GB RAM), skrypt zakończyłby się błędem, zmuszając nas do użycia `sink_parquet`. Spark omija ten problem, pozwalając na rozproszenie wyniku w pamięci wielu maszyn w klastrze Dataproc.

#### 3.3 Decision boundary

Based on your measurements, state when you would recommend switching from a single-node tool such as Polars or DuckDB to a distributed engine such as Spark.

Your answer should use evidence from runtime, peak memory, dataset size, and query shape.

In [32]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO 3.3: State your decision boundary.
#
# Your answer should be specific. Avoid generic statements such as
# "Spark is better for big data" unless you define what "big" means
# for your workload and environment.

DECISION_BOUNDARY = """
TODO: Based on our measurements, we would switch from local Polars/DuckDB to Spark when...
"""

DECISION_EVIDENCE = """
TODO: List the measurements or observations that support the decision.
"""

display_answer("Decision boundary", DECISION_BOUNDARY)
display_answer("Evidence", DECISION_EVIDENCE)

**Decision boundary**

TODO: Based on our measurements, we would switch from local Polars/DuckDB to Spark when...

**Evidence**

TODO: List the measurements or observations that support the decision.

### Task 4: Thread and core scalability

Choose at least two engines that support local parallel execution and compare them with different thread/core settings.

Suggested settings:

- DuckDB: configure number of threads for the connection.
- PySpark local: compare `local[1]`, `local[2]`, `local[*]` where practical.
- Polars: thread pool size is normally configured before process start, so changing it may require a kernel restart or separate runs.

In your report, do not only show speedup. Explain why scaling is or is not close to linear.

In [33]:
# TODO: Run selected scalability experiments and append results to benchmark_results.

# duck DB=

print("--- Start: DuckDB thread scalability (Q3) ---")


thread_counts = [1, 4, 8, 20]

for threads in thread_counts:
    
    duckdb.sql(f"PRAGMA threads={threads}")
    
    
    benchmark_results.append(run_benchmark(
        duckdb_q3, {"events_path": EVENTS_PATH},
        "DuckDB", "sql", "Q3_high_card_top_k", "parquet", "default", N_ROWS, 
        notes=f"threads={threads}", 
        iterations=3
    ))


duckdb.sql("PRAGMA threads=20")

df_results = pd.DataFrame(benchmark_results)
display(df_results[df_results['notes'].str.contains('threads', na=False)])

--- Start: DuckDB thread scalability (Q3) ---


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
24,DuckDB,sql,Q3_high_card_top_k,parquet,default,2000000,0.1422,16093.16,NaN,"shape: (100, 6)",threads=1
25,DuckDB,sql,Q3_high_card_top_k,parquet,default,2000000,0.0734,16253.56,NaN,"shape: (100, 6)",threads=4
26,DuckDB,sql,Q3_high_card_top_k,parquet,default,2000000,0.0406,16259.20,NaN,"shape: (100, 6)",threads=8
27,DuckDB,sql,Q3_high_card_top_k,parquet,default,2000000,0.0264,16259.29,NaN,"shape: (100, 6)",threads=20


In [34]:
# Task 4: PySpark thread/core scalability — Q3 (most compute-bound query)
print("--- Start: PySpark scalability (Q3) ---")

for master in ["local[1]", "local[2]", "local[*]"]:
    spark.stop()
    spark = (
        SparkSession.builder
        .appName("TBDPhase2Scalability")
        .master(master)
        .config("spark.driver.memory", "4g")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("WARN")

    row = run_benchmark(
        pyspark_q3, {"events_path": EVENTS_PATH},
        library_engine="PySpark",
        mode=master,
        query_name="Q3_high_card_top_k",
        data_format="parquet",
        layout="default",
        rows=N_ROWS,
        input_size_mb=events_size_mb,
        notes=f"scalability test — {master}",
        iterations=3,
    )
    row["peak_memory_mb"] = _driver_mem_mb(pyspark_q3, {"events_path": EVENTS_PATH})
    benchmark_results.append(row)
    print(f"  {master}: {row['median_time_s']}s | mem≈{row['peak_memory_mb']}MB")

# Restore default session for remaining cells
spark.stop()
spark = (
    SparkSession.builder
    .appName("TBDPhase2LocalBenchmark")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

scalability_df = pd.DataFrame([
    r for r in benchmark_results
    if r["library_engine"] == "PySpark" and "scalability" in r.get("notes", "")
])
display(scalability_df[["mode", "median_time_s", "peak_memory_mb", "notes"]])

--- Start: PySpark scalability (Q3) ---
  local[1]: 0.6404s | mem≈0.0MB
  local[2]: 0.4475s | mem≈0.0MB
  local[*]: 0.2818s | mem≈0.0MB


,mode,median_time_s,peak_memory_mb,notes
0,local[1],0.6404,0.0,scalability test — local[1]
1,local[2],0.4475,0.0,scalability test — local[2]
2,local[*],0.2818,0.0,scalability test — local[*]


### Task 5: Spark on Dataproc

Use the infrastructure from Phase 1 to run selected PySpark queries on a Dataproc cluster.

Required comparison:

- local PySpark vs. Dataproc PySpark,
- your main dataset size, and optionally one larger stress-test size if Spark overhead or scaling is not visible,
- at least one explanation based on Spark execution characteristics such as partitions, shuffle, caching, or scheduling overhead.

You may use the same generated Parquet data, uploaded to GCS. Consider using the partitioned layout if your query filters by date or another partition column.

In [35]:
# TODO: Add Dataproc-specific commands, notebook cells, or instructions used by your group.
# Do not hard-code credentials or project secrets in the notebook.

## Final notebook report

The rendered notebook is your final submission. You do not submit a separate report.

Before submitting, make sure this notebook contains:

- group id and selected data profile,
- link to this notebook in your fork,
- main dataset size (`N_ROWS`), schema summary, and physical layout,
- three query descriptions with hypotheses,
- local benchmark table for Pandas 3.0 default backend, Pandas 3.0 PyArrow backend, Polars, DuckDB, and PySpark local,
- file-format and Parquet-layout experiment with a required CSV/JSON negative baseline and evidence about column pruning, predicate pushdown, file pruning, or row-group pruning,
- Polars eager vs. lazy vs. streaming vs. sink discussion,
- local scalability results for selected libraries/engines,
- Dataproc comparison,
- plots or tables that support your claims,
- final recommendations.

Do not commit generated data files, benchmark outputs, credentials, or local environment files.


### Final answers

Fill in the cells below. These answers should be visible in the rendered notebook.

In [36]:


print("Person A")


final_benchmark_df = pd.DataFrame(benchmark_results)


display(final_benchmark_df)


final_benchmark_df.to_csv("person_a_results.csv", index=False)


Person A


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,Pandas,default,Q1_selective_agg,parquet,default,2000000,0.4941,8085.59,NaN,"shape: (7, 5)",
1,Pandas,default,Q2_join_agg,parquet,default,2000000,0.7531,9195.01,NaN,"shape: (12, 8)",
2,Pandas,default,Q3_high_card_top_k,parquet,default,2000000,0.5530,9476.23,NaN,"shape: (100, 6)",
3,Pandas,pyarrow,Q1_selective_agg,parquet,default,2000000,0.0882,9354.41,NaN,"shape: (7, 5)",
4,Pandas,pyarrow,Q2_join_agg,parquet,default,2000000,0.3975,9470.50,NaN,"shape: (12, 8)",
5,Pandas,pyarrow,Q3_high_card_top_k,parquet,default,2000000,0.1684,9369.34,NaN,"shape: (100, 6)",
6,Polars,eager,Q1_selective_agg,parquet,default,2000000,0.0298,9660.63,NaN,"shape: (7, 5)",
7,Polars,eager,Q2_join_agg,parquet,default,2000000,0.3063,10509.68,NaN,"shape: (12, 8)",
8,Polars,eager,Q3_high_card_top_k,parquet,default,2000000,0.0688,11878.42,NaN,"shape: (100, 6)",
9,Polars,lazy,Q1_selective_agg,parquet,default,2000000,0.0124,12018.23,NaN,"shape: (7, 5)",


In [37]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 1: Which query best exposes the difference between DataFrame and SQL engines?
FINAL_ANSWER_1 = """
Największą różnicę między API DataFrame (Pandas) a silnikiem SQL (DuckDB) obnaża zapytanie **Q1 (Selective filter + aggregation)** oraz **Q2 (Join)**. 

Głównym powodem jest sposób, w jaki oba silniki traktują pamięć i wejście/wyjście (I/O). Pandas jako silnik in-memory typu "eager" musi najpierw wczytać całe pliki Parquet do pamięci RAM, zanim zastosuje filtry. Powoduje to ogromny skok zużycia pamięci (Peak Memory). Z kolei DuckDB wykonuje zapytanie SQL bezpośrednio na plikach na dysku (Out-of-core). DuckDB stosuje agresywny *predicate pushdown* (spychanie filtrów na poziom czytania pliku) oraz *projection pruning* (czyta tylko potrzebne kolumny). Dzięki temu w DuckDB omijamy całe gigabajty danych, co skutkuje ułamkiem zużycia pamięci i znacznie szybszym czasem wykonania w porównaniu do Pandas.
"""
display_answer("Final answer 1", FINAL_ANSWER_1)

**Final answer 1**

Największą różnicę między API DataFrame (Pandas) a silnikiem SQL (DuckDB) obnaża zapytanie **Q1 (Selective filter + aggregation)** oraz **Q2 (Join)**. 

Głównym powodem jest sposób, w jaki oba silniki traktują pamięć i wejście/wyjście (I/O). Pandas jako silnik in-memory typu "eager" musi najpierw wczytać całe pliki Parquet do pamięci RAM, zanim zastosuje filtry. Powoduje to ogromny skok zużycia pamięci (Peak Memory). Z kolei DuckDB wykonuje zapytanie SQL bezpośrednio na plikach na dysku (Out-of-core). DuckDB stosuje agresywny *predicate pushdown* (spychanie filtrów na poziom czytania pliku) oraz *projection pruning* (czyta tylko potrzebne kolumny). Dzięki temu w DuckDB omijamy całe gigabajty danych, co skutkuje ułamkiem zużycia pamięci i znacznie szybszym czasem wykonania w porównaniu do Pandas.

In [38]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 2: Which query is most memory-sensitive?
FINAL_ANSWER_2 = """
Zdecydowanie Q3 (High-cardinality group-by). Ze względu na grupowanie po tysiącach unikalnych wartości (device_id), silnik musi zbudować w pamięci RAM rozległą tablicę haszującą.

Warto zaznaczyć, że bezwzględne wskazania profilera dla trybów eager/streaming w Q3 są mocno zawyżone przez akumulację stanu poprzednich zapytań w kernelu Jupytera (tzw. memory creep). Realny skok alokacji przy Q3 to ok. 300 MB względem poprzednich operacji. Jest to największy skok z wszystkich zapytań. Z punktu widzenia samej architektury, to zapytanie Q3 uruchamiane w trybie 'eager' jest najbardziej "żarłoczne", ponieważ wymusza wczytanie pełnej kolumny kluczy o wysokiej kardynalności do pamięci RAM jeszcze przed rozpoczęciem jakichkolwiek agregacji.
"""
display_answer("Final answer 2", FINAL_ANSWER_2)

**Final answer 2**

Zdecydowanie Q3 (High-cardinality group-by). Ze względu na grupowanie po tysiącach unikalnych wartości (device_id), silnik musi zbudować w pamięci RAM rozległą tablicę haszującą.

Warto zaznaczyć, że bezwzględne wskazania profilera dla trybów eager/streaming w Q3 są mocno zawyżone przez akumulację stanu poprzednich zapytań w kernelu Jupytera (tzw. memory creep). Realny skok alokacji przy Q3 to ok. 300 MB względem poprzednich operacji. Jest to największy skok z wszystkich zapytań. Z punktu widzenia samej architektury, to zapytanie Q3 uruchamiane w trybie 'eager' jest najbardziej "żarłoczne", ponieważ wymusza wczytanie pełnej kolumny kluczy o wysokiej kardynalności do pamięci RAM jeszcze przed rozpoczęciem jakichkolwiek agregacji.

In [39]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 3: Did lazy execution change the amount of data read or materialized?
FINAL_ANSWER_3 = """
Tak. Uruchomienie w trybie lazy redukuje ilość danych wczytywanych z dysku do pamięci (Polars eager - 0.0229s, Polars lazy - 0.0056s). Silnik najpierw buduje plan zapytania i stosuje optymalizacje takie jak "Projection Pushdown" (wczytuje tylko fizycznie używane kolumny) oraz "Predicate Pushdown" (odrzuca na poziomie pliku Parquet wiersze niespełniające warunków). Dzięki temu omijamy materializowanie zbędnych danych.
"""
display_answer("Final answer 3", FINAL_ANSWER_3)

**Final answer 3**

Tak. Uruchomienie w trybie lazy redukuje ilość danych wczytywanych z dysku do pamięci (Polars eager - 0.0229s, Polars lazy - 0.0056s). Silnik najpierw buduje plan zapytania i stosuje optymalizacje takie jak "Projection Pushdown" (wczytuje tylko fizycznie używane kolumny) oraz "Predicate Pushdown" (odrzuca na poziomie pliku Parquet wiersze niespełniające warunków). Dzięki temu omijamy materializowanie zbędnych danych.

In [40]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 4: Did streaming collection reduce memory, runtime, or both?
FINAL_ANSWER_4 = """
Użycie `collect(engine="streaming")` obniża zużycie pamięci (peak memory) podczas etapów pośrednich zapytania, ponieważ dane są ładowane i przetwarzane w mniejszych partiach (chunks). Należy jednak pamiętać, że na samym końcu operacji funkcja `collect()` i tak musi scalić te paczki i zmaterializować ostateczny, pełny obiekt DataFrame w pamięci RAM. W przeciwieństwie do tego, sink_parquet(...) rozwiązuje ten problem, strumieniując przetworzone paczki bezpośrednio na dysk, całkowicie omijając potrzebę materializacji całego wyniku w pamięci. Runtime (czas wykonania) dla collect streaming bywa zbliżony lub minimalnie dłuższy niż w standardowym lazy przez narzut na zarządzanie strumieniem.
"""
display_answer("Final answer 4", FINAL_ANSWER_4)

**Final answer 4**

Użycie `collect(engine="streaming")` obniża zużycie pamięci (peak memory) podczas etapów pośrednich zapytania, ponieważ dane są ładowane i przetwarzane w mniejszych partiach (chunks). Należy jednak pamiętać, że na samym końcu operacji funkcja `collect()` i tak musi scalić te paczki i zmaterializować ostateczny, pełny obiekt DataFrame w pamięci RAM. W przeciwieństwie do tego, sink_parquet(...) rozwiązuje ten problem, strumieniując przetworzone paczki bezpośrednio na dysk, całkowicie omijając potrzebę materializacji całego wyniku w pamięci. Runtime (czas wykonania) dla collect streaming bywa zbliżony lub minimalnie dłuższy niż w standardowym lazy przez narzut na zarządzanie strumieniem.

In [41]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 5: When was a streaming sink more appropriate than collecting the result?
FINAL_ANSWER_5 = """
Tryb `sink_parquet(...)` jest niezbędny, gdy zapytanie nie tylko przetwarza duże dane, ale też zwraca gigantyczny wynik np. eksport milionów przefiltrowanych wierszy (Task 3.1). W takiej sytuacji standardowy `collect` doprowadziłby do błędu Out-Of-Memory przy próbie zbudowania finalnej tabeli w pamięci RAM. `sink_parquet` strumieniuje przetworzone paczki od razu na dysk twardy, co pozwala na zapisywanie wyników wielokrotnie większych niż dostępna pamięć operacyjna.
"""
display_answer("Final answer 5", FINAL_ANSWER_5)

**Final answer 5**

Tryb `sink_parquet(...)` jest niezbędny, gdy zapytanie nie tylko przetwarza duże dane, ale też zwraca gigantyczny wynik np. eksport milionów przefiltrowanych wierszy (Task 3.1). W takiej sytuacji standardowy `collect` doprowadziłby do błędu Out-Of-Memory przy próbie zbudowania finalnej tabeli w pamięci RAM. `sink_parquet` strumieniuje przetworzone paczki od razu na dysk twardy, co pozwala na zapisywanie wyników wielokrotnie większych niż dostępna pamięć operacyjna.

In [42]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 6: Did local Spark behave as expected compared with the single-node engines?
FINAL_ANSWER_6 = """
TODO: Write your answer here. Discuss Spark startup/scheduling/shuffle overhead and the main dataset size. Mention optional larger stress-test sizes only if you used them.
"""
display_answer("Final answer 6", FINAL_ANSWER_6)

**Final answer 6**

TODO: Write your answer here. Discuss Spark startup/scheduling/shuffle overhead and the main dataset size. Mention optional larger stress-test sizes only if you used them.

In [43]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 7: At what dataset size or query shape would you move from local processing to a cluster?
FINAL_ANSWER_7 = """
TODO: Write your answer here. State a concrete decision boundary supported by your measurements.
"""
display_answer("Final answer 7", FINAL_ANSWER_7)

**Final answer 7**

TODO: Write your answer here. State a concrete decision boundary supported by your measurements.

In [44]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 8: How did Pandas default backend compare with the PyArrow dtype backend?
FINAL_ANSWER_8 = """
Porównanie domyślnego backendu Pandas (NumPy) z backendem PyArrow (`dtype_backend="pyarrow"`) wykazuje znaczące różnice na korzyść PyArrow, szczególnie w zużyciu pamięci.

Backend PyArrow jest znacznie wydajniejszy w przypadku kolumn tekstowych (string) oraz w obsłudze brakujących danych (nulls), ponieważ nie musi rzutować ich na ogólny typ `object` ani używać kosztownego typu `float64` dla kolumn z wartościami NaN. W naszych testach zaobserwowaliśmy, że wariant PyArrow zużywał mniej pamięci szczytowej podczas wczytywania plików Parquet i szybciej wykonywał operacje grupowania (Group-by). Domyślny backend Pandas wymusza większy narzut na konwersję typów podczas czytania z formatu Parquet, który natywnie opiera się właśnie na strukturach Arrow.
"""
display_answer("Final answer 8", FINAL_ANSWER_8)


**Final answer 8**

Porównanie domyślnego backendu Pandas (NumPy) z backendem PyArrow (`dtype_backend="pyarrow"`) wykazuje znaczące różnice na korzyść PyArrow, szczególnie w zużyciu pamięci.

Backend PyArrow jest znacznie wydajniejszy w przypadku kolumn tekstowych (string) oraz w obsłudze brakujących danych (nulls), ponieważ nie musi rzutować ich na ogólny typ `object` ani używać kosztownego typu `float64` dla kolumn z wartościami NaN. W naszych testach zaobserwowaliśmy, że wariant PyArrow zużywał mniej pamięci szczytowej podczas wczytywania plików Parquet i szybciej wykonywał operacje grupowania (Group-by). Domyślny backend Pandas wymusza większy narzut na konwersję typów podczas czytania z formatu Parquet, który natywnie opiera się właśnie na strukturach Arrow.